# Results — Main Analysis Notebook

Structured to mirror the thesis results chapter (Section 7):

- **§7.1** Silver Dataset Quality Analysis
- **Comparison I** Teacher vs. Student (silver test set, n=1847)
- **Comparison II** Student vs. Student — LLaMA vs. Mistral
- **Comparison III** Student vs. Human (gold standard, n=50)

> Calibration diagrams and per-model confidence distributions already live in `confidence_analysis.ipynb`.

In [ ]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as scipy_stats
from sklearn.metrics import (
    cohen_kappa_score, f1_score, precision_score, recall_score, accuracy_score
)
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
_PAL = sns.color_palette('Set2', 6)
MODEL_COLORS = {
    'LLaMA Vanilla':      _PAL[0],
    'Mistral Vanilla':    _PAL[1],
    'LLaMA Fine-tuned':   _PAL[2],
    'Mistral Fine-tuned': _PAL[3],
    'Teacher (PoC)':      _PAL[4],
}

def _find_project_root() -> Path:
    """Locate Thesis_IT_TicketClassification/ regardless of kernel CWD."""
    candidates = []

    # Strategy 1: VS Code Jupyter injects the notebook's absolute path here
    if '__vsc_ipynb_file__' in globals():
        p = Path(globals()['__vsc_ipynb_file__']).resolve().parents[2]
        if (p / 'data').is_dir() and (p / 'Notebooks').is_dir():
            candidates.append(p)

    # Strategy 2: walk up from CWD looking for data/results + Notebooks
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'data' / 'results').is_dir() and (p / 'Notebooks').is_dir():
            candidates.append(p)
            break

    # Strategy 3: walk up from CWD looking for the known sub-path
    for p in [Path.cwd()] + list(Path.cwd().parents):
        candidate = p / 'Users' / 'josta' / 'Thesis_IT_TicketClassification'
        if candidate.is_dir() and (candidate / 'data').is_dir():
            candidates.append(candidate)
            break

    if candidates:
        return candidates[0]

    raise RuntimeError(
        "Cannot locate project root.\n"
        "Add this line before the ROOT assignment:\n"
        "  ROOT = Path('/absolute/path/to/Thesis_IT_TicketClassification')"
    )

ROOT    = Path(_find_project_root())
DATA    = ROOT / 'data'
RES     = DATA / 'results'
LABELED = DATA / 'labeled'
OUT     = ROOT / 'Notebooks' / 'Results Analysis'
print(f'ROOT: {ROOT}')
print(f'RES:  {RES}  (exists={RES.is_dir()})')

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
_LABEL_RE = re.compile(r'\(([^,]+),\s*([^)]+)\),\s*(.+)')

def parse_levels(label):
    """Return (L1, L2, L3) from label string, or (None, None, None)."""
    if not isinstance(label, str): return None, None, None
    m = _LABEL_RE.match(label.strip())
    return (m.group(1).strip(), m.group(2).strip(), m.group(3).strip()) if m else (None, None, None)

def add_levels(df, col, prefix):
    df[[f'{prefix}_l1', f'{prefix}_l2', f'{prefix}_l3']] = (
        df[col].apply(lambda x: pd.Series(parse_levels(x),
                                           index=[f'{prefix}_l1', f'{prefix}_l2', f'{prefix}_l3']))
    )
    return df

def load_result(path):
    with open(path) as f:
        raw = json.load(f)
    df = pd.DataFrame(raw['data'], columns=raw['columns'])
    for c in ['correct', 'is_fallback', 'format_compliant']:
        df[c] = df[c].astype(bool)
    df['number'] = df['text'].str.extract(r'(INC\d+)', expand=False)
    add_levels(df, 'true_label',      'true')
    add_levels(df, 'predicted_label', 'pred')
    df['correct_l1'] = df['true_l1'] == df['pred_l1']
    df['correct_l2'] = df['true_l2'] == df['pred_l2']
    df['correct_l3'] = df['true_l3'] == df['pred_l3']
    return df

def scalar_from_wandb(path):
    """Read a single-row W&B export and return the metric value."""
    df = pd.read_csv(path)
    val_col = [c for c in df.columns if 'MIN' not in c and 'MAX' not in c and c != 'Step'][0]
    return float(df[val_col].iloc[0])

In [ ]:
# ── Load all result files ─────────────────────────────────────────────────────
MODEL_ORDER = ['LLaMA Vanilla', 'Mistral Vanilla', 'LLaMA Fine-tuned', 'Mistral Fine-tuned']
MODEL_FILES = {
    'LLaMA Vanilla':      RES / 'vanilla_llama.json',
    'Mistral Vanilla':    RES / 'vanilla_ministral.json',
    'LLaMA Fine-tuned':   RES / 'qlora_llama.json',
    'Mistral Fine-tuned': RES / 'qlora_mistral.json',
}
dfs = {name: load_result(path) for name, path in MODEL_FILES.items()}
for name, df in dfs.items():
    print(f'{name:25s}  n={len(df)}  acc={df["correct"].mean():.3f}')

# ── Gold prediction files (student models on 50 gold tickets) ─────────────────
gold = {
    'LLaMA Fine-tuned':   load_result(RES / 'predictions_llama_gold.json'),
    'Mistral Fine-tuned': load_result(RES / 'predictions_mistral_gold.json'),
}

# ── Human gold labels ─────────────────────────────────────────────────────────
human_gold = pd.read_csv(LABELED / 'final_tags_human.csv')
human_gold['human_label'] = (
    human_gold['category final'].str.strip() + ', ' +
    human_gold['Scenario final'].str.strip()
)
add_levels(human_gold, 'human_label', 'human')

# ── Teacher (PoC) on the 50 gold tickets ─────────────────────────────────────
poc = pd.read_csv(LABELED / 'PoC/PoC_results.csv')
add_levels(poc, 'label', 'poc')
poc = poc.merge(human_gold[['number','human_label','human_l1','human_l2','human_l3']], on='number', how='left')
poc['vs_human_full'] = poc['label'].str.strip() == poc['human_label'].str.strip()
poc['vs_human_l1']   = poc['poc_l1'] == poc['human_l1']
poc['vs_human_l2']   = poc['poc_l2'] == poc['human_l2']
poc['vs_human_l3']   = poc['poc_l3'] == poc['human_l3']

# Align gold predictions to human labels via PoC number lookup
poc_key = poc[['number','text']].copy()
poc_key['text_key'] = poc_key['text'].str[:80].str.strip()
for gdf in gold.values():
    gdf['text_key'] = gdf['text'].str[:80].str.strip()
    gdf.update(gdf.merge(poc_key, on='text_key', how='left', suffixes=('','_poc')))
    gdf.merge(poc_key[['text_key','number']], on='text_key', how='left').pipe(
        lambda d: gdf.__setitem__('number', d['number_y'].combine_first(gdf['number']))
    )

for name, gdf in gold.items():
    if 'number' not in gdf.columns or gdf['number'].isna().all():
        merged_tmp = gdf.merge(poc_key[['text_key','number']], on='text_key', how='left')
        gdf['number'] = merged_tmp['number'].values
    gdf = gdf.merge(
        human_gold[['number','human_label','human_l1','human_l2','human_l3']],
        on='number', how='left'
    )
    gdf['vs_human_full'] = gdf['predicted_label'].str.strip() == gdf['human_label'].str.strip()
    gdf['vs_human_l1']   = gdf['pred_l1'] == gdf['human_l1']
    gdf['vs_human_l2']   = gdf['pred_l2'] == gdf['human_l2']
    gdf['vs_human_l3']   = gdf['pred_l3'] == gdf['human_l3']
    gold[name] = gdf

print('Gold LLaMA human match:',   gold['LLaMA Fine-tuned']['human_label'].notna().sum(), '/50')
print('Gold Mistral human match:', gold['Mistral Fine-tuned']['human_label'].notna().sum(), '/50')

In [ ]:
# ── Load silver standard (for S* analysis and S* correlation) ─────────────────
silver = pd.read_csv(
    LABELED / 'CISC_Fixed/silver_standard_dataset.csv',
    usecols=['number','label','scientific_confidence','max_path_confidence','consistency']
)
print(f'Silver standard: {len(silver)} rows')

# Join S* scores into main result DataFrames
for name, df in dfs.items():
    dfs[name] = df.merge(
        silver[['number','scientific_confidence']].rename(columns={'scientific_confidence':'s_star'}),
        on='number', how='left'
    )
print('S* join coverage (fine-tuned LLaMA):',
      dfs['LLaMA Fine-tuned']['s_star'].notna().sum(), '/', len(dfs['LLaMA Fine-tuned']))

---
## §7.1 Silver Dataset Quality Analysis

In [ ]:
# ── S* distribution stats ─────────────────────────────────────────────────────
s = silver['scientific_confidence']
print('S* distribution:')
print(f'  Mean:   {s.mean():.4f}')
print(f'  Median: {s.median():.4f}')
print(f'  Std:    {s.std():.4f}')
print(f'  >= 0.75 (filter threshold): {(s >= 0.75).mean():.1%} of total corpus')
print()

# Label distribution L1
silver_l1 = silver['label'].apply(lambda x: parse_levels(x)[0])
l1_dist = silver_l1.value_counts()
print('L1 label distribution:')
display(l1_dist.to_frame('count').assign(pct=lambda d: (d['count']/len(silver)*100).round(2)))

In [ ]:
# ── Visual 7.1a: S* distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# KDE
ax = axes[0]
silver['scientific_confidence'].plot.kde(ax=ax, color=_PAL[4], lw=2)
ax.axvline(0.75, color='red', lw=1.5, ls='--', label='S* ≥ 0.75 filter')
ax.axvline(silver['scientific_confidence'].mean(), color='navy', lw=1.5, ls=':', label=f'Mean={silver["scientific_confidence"].mean():.3f}')
ax.set_xlabel('Scientific Confidence (S*)')
ax.set_title('S* Distribution (Silver Dataset)', fontweight='bold')
ax.legend(fontsize=9)

# L1 label distribution bar
ax = axes[1]
l1_dist.plot.bar(ax=ax, color=sns.color_palette('tab10', len(l1_dist)), edgecolor='white')
ax.set_title('Label Distribution — Level 1\n(Silver Dataset)', fontweight='bold')
ax.set_ylabel('Ticket count')
ax.tick_params(axis='x', rotation=30)
ax.set_xlabel('')

# Full-label long-tail (log scale)
ax = axes[2]
full_label_counts = silver['label'].value_counts()
ax.plot(range(len(full_label_counts)), full_label_counts.values, color=_PAL[4], lw=1.5)
ax.set_yscale('log')
ax.set_xlabel('Label rank (by frequency)')
ax.set_ylabel('Count (log scale)')
ax.set_title('Full-Label Long Tail\n(Silver Dataset)', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT / 'silver_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fallback analysis on silver standard ──────────────────────────────────────
silver_l1_col = silver['label'].apply(lambda x: parse_levels(x)[0])
fallback_mask = silver_l1_col == '1 Misc incidents'
print(f'Fallback (Misc incidents) in silver dataset: '
      f'{fallback_mask.sum()} / {len(silver)} = {fallback_mask.mean():.2%}')
print()

# S* distribution for fallback vs non-fallback tickets
print('Mean S* — fallback vs non-fallback:')
print(silver.groupby(fallback_mask)['scientific_confidence'].describe().round(4))

---
## Comparison I: Teacher vs. Student
### Classification Fidelity (1847 silver-standard test tickets)

In [ ]:
# ── Table I-A: primary metrics ────────────────────────────────────────────────
def classification_metrics(df):
    y_t = df['true_label'].astype(str)
    y_p = df['predicted_label'].astype(str)
    return {
        'Accuracy (full)':  round(df['correct'].mean(), 4),
        'Accuracy L1':      round(df['correct_l1'].mean(), 4),
        'Accuracy L2':      round(df['correct_l2'].mean(), 4),
        'Accuracy L3':      round(df['correct_l3'].mean(), 4),
        'Macro F1':         round(f1_score(y_t, y_p, average='macro',     zero_division=0), 4),
        'Weighted F1':      round(f1_score(y_t, y_p, average='weighted',  zero_division=0), 4),
        'Macro Precision':  round(precision_score(y_t, y_p, average='macro',    zero_division=0), 4),
        'Macro Recall':     round(recall_score(y_t, y_p,    average='macro',    zero_division=0), 4),
        'Format Compliance': round(df['format_compliant'].mean(), 4),
        'Fallback Rate':    round(df['is_fallback'].mean(), 4),
        'Mean Confidence':  round(df['student_confidence'].mean(), 4),
    }

metrics_table = pd.DataFrame(
    {name: classification_metrics(df) for name, df in dfs.items()}
).T.loc[MODEL_ORDER]

print('=== Table I-A: Classification metrics — all 4 models (n=1847) ===')
display(metrics_table)

In [ ]:
# ── Visual I-1: Hierarchical accuracy bar chart ───────────────────────────────
levels     = ['Accuracy L1', 'Accuracy L2', 'Accuracy L3', 'Accuracy (full)']
level_lbls = ['Level 1\n(Service type)', 'Level 2\n(Product)', 'Level 3\n(Scenario)', 'Full label\n(exact match)']
x      = np.arange(len(levels))
n_mdls = len(MODEL_ORDER)
width  = 0.18
offsets = np.linspace(-(n_mdls-1)/2, (n_mdls-1)/2, n_mdls) * width

fig, ax = plt.subplots(figsize=(13, 6))
for name, offset in zip(MODEL_ORDER, offsets):
    vals = [metrics_table.loc[name, lv] for lv in levels]
    bars = ax.bar(x + offset, vals, width, label=name,
                  color=MODEL_COLORS[name], edgecolor='white', lw=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.0%}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(level_lbls, fontsize=11)
ax.set_ylabel('Accuracy')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.13)
ax.set_title('Hierarchical Accuracy — All 4 Models (n=1847)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower left')
plt.tight_layout()
plt.savefig(OUT / 'hierarchical_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Table I-B: Per-category F1 (fine-tuned models) ───────────────────────────
def per_l1_f1(df, model_name, min_support=5):
    rows = []
    for cat in sorted(df['true_l1'].dropna().unique()):
        sub = df[df['true_l1'] == cat]
        if len(sub) < min_support: continue
        y_t = sub['true_label'].astype(str)
        y_p = sub['predicted_label'].astype(str)
        labels = sorted(y_t.unique())
        rows.append({
            'L1 Category': cat,
            'Model': model_name,
            'Precision':    round(precision_score(y_t, y_p, labels=labels, average='macro', zero_division=0), 4),
            'Recall':       round(recall_score(y_t, y_p,    labels=labels, average='macro', zero_division=0), 4),
            'Macro F1':     round(f1_score(y_t, y_p,        labels=labels, average='macro', zero_division=0), 4),
            'Weighted F1':  round(f1_score(y_t, y_p,        labels=labels, average='weighted', zero_division=0), 4),
            'Accuracy':     round(sub['correct'].mean(), 4),
            'Support':      len(sub),
        })
    return pd.DataFrame(rows)

ft_models = ['LLaMA Fine-tuned', 'Mistral Fine-tuned']
f1_all = pd.concat([per_l1_f1(dfs[m], m) for m in ft_models], ignore_index=True)

print('=== Table I-B: Per-L1 F1 — Fine-tuned models ===')
display(
    f1_all.pivot_table(index='L1 Category', columns='Model',
                       values=['Weighted F1','Macro F1','Accuracy','Support'])
    .round(4)
)

In [ ]:
# ── Visual I-2: Per-category weighted F1 bar chart ────────────────────────────
pivot_f1 = (
    f1_all.pivot(index='L1 Category', columns='Model', values='Weighted F1')[ft_models]
    .fillna(0)
    .loc[lambda d: d.mean(axis=1).sort_values(ascending=False).index]
)
fig, ax = plt.subplots(figsize=(14, 6))
pivot_f1.plot.bar(ax=ax, color=[MODEL_COLORS[m] for m in ft_models],
                  edgecolor='white', width=0.7)
ax.set_title('Per-Category Weighted F1 — Fine-tuned Models (n=1847)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Weighted F1')
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Model', fontsize=10)
plt.tight_layout()
plt.savefig(OUT / 'per_category_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fallback rate analysis ────────────────────────────────────────────────────
print('=== Fallback rates ===')
for name, df in dfs.items():
    fb_rate = df['is_fallback'].mean()
    fb_acc  = df.loc[df['is_fallback'], 'correct'].mean()
    nfb_acc = df.loc[~df['is_fallback'], 'correct'].mean()
    print(f'{name:25s}  fallback={fb_rate:.2%}  '
          f'acc(fallback)={fb_acc:.3f}  acc(normal)={nfb_acc:.3f}')

### Misclassification Analysis

In [ ]:
# ── Top L1 misclassification pairs ────────────────────────────────────────────
for name in ft_models:
    df = dfs[name]
    wrong = df[df['true_l1'] != df['pred_l1']].copy()
    wrong['pair'] = wrong['true_l1'] + '  →  ' + wrong['pred_l1']
    pairs = wrong['pair'].value_counts().head(5)
    print(f'\n=== Top 5 L1 misclassification pairs — {name} ===')
    display(pairs.to_frame('Count'))

In [ ]:
# ── Visual I-3: L1 confusion matrix heatmap ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, name in zip(axes, ft_models):
    df = dfs[name]
    top_cats = df['true_l1'].value_counts().head(8).index.tolist()
    sub = df[df['true_l1'].isin(top_cats)]
    pred_cats = sub['pred_l1'].where(sub['pred_l1'].isin(top_cats), other='Other')
    conf = pd.crosstab(sub['true_l1'], pred_cats)
    for cat in top_cats:
        if cat not in conf.columns: conf[cat] = 0
    conf = conf.reindex(index=top_cats).fillna(0)
    conf_norm = conf.div(conf.sum(axis=1), axis=0)

    sns.heatmap(conf_norm, annot=True, fmt='.0%', cmap='Blues',
                linewidths=0.4, ax=ax, vmin=0, vmax=1,
                cbar_kws={'label': 'Row-normalised recall'})
    ax.set_title(f'L1 Confusion Matrix — {name}\n(top-8 true classes, row-normalised)',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted L1')
    ax.set_ylabel('True L1')
    ax.tick_params(axis='x', rotation=35, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(OUT / 'confusion_matrix_l1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error typology (mirroring human inter-rater analysis) ─────────────────────
# Granular symptom: correct L1+L2, wrong L3
# Subcategory:      correct L1, wrong L2
# Cross-domain:     wrong L1 entirely
# Fallback retreat: predicted Misc incidents

for name in ft_models:
    df = dfs[name]
    wrong = df[~df['correct']]
    n = len(wrong)
    granular  = ((wrong['correct_l1']) & (wrong['correct_l2']) & (~wrong['correct_l3'])).sum()
    subcat    = ((wrong['correct_l1']) & (~wrong['correct_l2'])).sum()
    crossdom  = (~wrong['correct_l1']).sum()
    fallback  = (wrong['pred_l1'] == '1 Misc incidents').sum()
    print(f'\n=== Error typology — {name} (n_wrong={n}) ===')
    print(f'  Granular symptom (L1✓ L2✓ L3✗): {granular:>4}  ({granular/n:.1%})')
    print(f'  Subcategory      (L1✓ L2✗):     {subcat:>4}  ({subcat/n:.1%})')
    print(f'  Cross-domain     (L1✗):          {crossdom:>4}  ({crossdom/n:.1%})')
    print(f'  Fallback retreat (→Misc):        {fallback:>4}  ({fallback/n:.1%})')

In [ ]:
# ── S* error concentration ────────────────────────────────────────────────────
# Are errors concentrated on low-S* tickets?
for name in ft_models:
    df = dfs[name].dropna(subset=['s_star'])
    correct_sstar = df.loc[df['correct'], 's_star'].mean()
    wrong_sstar   = df.loc[~df['correct'], 's_star'].mean()
    rho, pval = scipy_stats.spearmanr(df['s_star'], df['correct'].astype(int))
    print(f'{name}:')
    print(f'  Mean S* (correct)={correct_sstar:.4f}  Mean S* (wrong)={wrong_sstar:.4f}')
    print(f'  Spearman ρ(S*, correct)={rho:.4f}  p={pval:.4f}')
    print()

In [ ]:
# ── Visual I-4: S* vs student confidence scatter + error density ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name in zip(axes, ft_models):
    df = dfs[name].dropna(subset=['s_star'])
    colors = df['correct'].map({True: MODEL_COLORS[name], False: 'tomato'})
    ax.scatter(df['s_star'], df['student_confidence'],
               c=colors, alpha=0.35, s=12, edgecolors='none')
    rho, _ = scipy_stats.spearmanr(df['s_star'], df['student_confidence'])
    ax.set_title(f'{name}\nSpearman ρ(S*, student_conf) = {rho:.3f}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Teacher S* score')
    ax.set_ylabel('Student confidence')
    ax.set_xlim(0.7, 1.02)
    ax.set_ylim(0, 1.02)
    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([0],[0], marker='o', color='w', markerfacecolor=MODEL_COLORS[name], ms=8, label='Correct'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato', ms=8, label='Incorrect'),
    ], fontsize=9)

plt.tight_layout()
plt.savefig(OUT / 'sstar_confidence_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### Generation Quality

In [ ]:
# ── ROUGE-L and BERTScore F1 (all 4 models) ───────────────────────────────────
# Each file is a single-row W&B export (one aggregate score per model)
gen_quality = pd.DataFrame([
    {'Model': 'LLaMA Vanilla',
     'ROUGE-L':    scalar_from_wandb(RES / 'rouge_llama_vanilla.csv'),
     'BERTScore F1': scalar_from_wandb(RES / 'bertscore_llama_vanilla.csv')},
    {'Model': 'Mistral Vanilla',
     'ROUGE-L':    scalar_from_wandb(RES / 'rouge_mistral_vanilla.csv'),
     'BERTScore F1': scalar_from_wandb(RES / 'bertscore_mistral_vanilla.csv')},
    {'Model': 'LLaMA Fine-tuned',
     'ROUGE-L':    scalar_from_wandb(RES / 'rouge_llama_finetuned.csv'),
     'BERTScore F1': scalar_from_wandb(RES / 'bertscore_llama_finetuned.csv')},
    {'Model': 'Mistral Fine-tuned',
     'ROUGE-L':    scalar_from_wandb(RES / 'rouge_mistral_finetuned.csv'),
     'BERTScore F1': scalar_from_wandb(RES / 'bertscore_mistral_finetuned.csv')},
]).set_index('Model').loc[MODEL_ORDER].round(4)

print('=== Generation quality — ROUGE-L and BERTScore F1 ===')
display(gen_quality)

In [ ]:
# ── Visual I-5: ROUGE-L and BERTScore grouped bar chart ───────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(MODEL_ORDER))
w = 0.35
bars_r = ax.bar(x - w/2, gen_quality['ROUGE-L'], w,
                color=[MODEL_COLORS[m] for m in MODEL_ORDER],
                label='ROUGE-L', edgecolor='white', alpha=0.9)
bars_b = ax.bar(x + w/2, gen_quality['BERTScore F1'], w,
                color=[MODEL_COLORS[m] for m in MODEL_ORDER],
                label='BERTScore F1', edgecolor='white', alpha=0.5, hatch='//')
for bar in list(bars_r) + list(bars_b):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, rotation=12, fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Reasoning Generation Quality\nROUGE-L and BERTScore F1 (vs. teacher reasoning)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT / 'generation_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Spearman ρ(S*, student_confidence) ───────────────────────────────────────
print('=== Spearman ρ(S*, student_confidence) — fine-tuned models ===')
for name in ft_models:
    df = dfs[name].dropna(subset=['s_star'])
    rho, pval = scipy_stats.spearmanr(df['s_star'], df['student_confidence'])
    print(f'{name:25s}  ρ={rho:.4f}  p={pval:.4e}  n={len(df)}')

---
## Comparison II: Student vs. Student — LLaMA vs. Mistral

In [ ]:
# ── Side-by-side metrics table ────────────────────────────────────────────────
cmp2 = metrics_table.loc[ft_models].T
print('=== Comparison II: Fine-tuned models side-by-side ===')
display(cmp2)

In [ ]:
# ── Do both models fail on the same tickets? ──────────────────────────────────
l_df = dfs['LLaMA Fine-tuned'].set_index('number')['correct']
m_df = dfs['Mistral Fine-tuned'].set_index('number')['correct']
common_idx = l_df.index.intersection(m_df.index)
l_c, m_c = l_df[common_idx], m_df[common_idx]

both_correct  = (l_c & m_c).sum()
only_llama    = (l_c & ~m_c).sum()
only_mistral  = (~l_c & m_c).sum()
both_wrong    = (~l_c & ~m_c).sum()
n = len(common_idx)

print(f'Shared tickets: n={n}')
print(f'Both correct:       {both_correct:>4} ({both_correct/n:.1%})')
print(f'Only LLaMA correct: {only_llama:>4} ({only_llama/n:.1%})')
print(f'Only Mistral correct:{only_mistral:>4} ({only_mistral/n:.1%})')
print(f'Both wrong:         {both_wrong:>4} ({both_wrong/n:.1%})')
print(f'\nAgreement rate (same outcome): {(both_correct+both_wrong)/n:.1%}')

In [ ]:
# ── ROUGE-L and BERTScore: fine-tuned comparison ──────────────────────────────
print('=== Generation quality — fine-tuned only ===')
display(gen_quality.loc[ft_models])

In [ ]:
# ── Mid-training curves ───────────────────────────────────────────────────────
def load_mid(metric_key, model_key):
    df = pd.read_csv(RES / f'mid_{metric_key}_{model_key}.csv')
    val_col = [c for c in df.columns if 'MIN' not in c and 'MAX' not in c and c != 'Step'][0]
    return df[['Step', val_col]].rename(columns={val_col: 'value'})

model_keys = {'LLaMA Fine-tuned': 'llama', 'Mistral Fine-tuned': 'mistral'}
metrics_map = {'Exact Match': 'exactmatch', 'Macro F1': 'macrof1', 'Weighted F1': 'weightedf1'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, (metric_label, metric_key) in zip(axes, metrics_map.items()):
    for name, mkey in model_keys.items():
        mid = load_mid(metric_key, mkey)
        ax.plot(mid['Step'], mid['value'], marker='o', lw=2,
                label=name, color=MODEL_COLORS[name])
    ax.set_title(metric_label, fontweight='bold')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

fig.suptitle('Mid-Training Evaluation Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'mid_training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Final / best values table
mid_rows = []
for name, mkey in model_keys.items():
    row = {'Model': name}
    for mlabel, mmetric in metrics_map.items():
        mid = load_mid(mmetric, mkey)
        row[f'{mlabel} (best)']  = round(mid['value'].max(), 4)
        row[f'{mlabel} (final)'] = round(mid['value'].iloc[-1], 4)
    mid_rows.append(row)
display(pd.DataFrame(mid_rows).set_index('Model'))

In [ ]:
# ── Latency ───────────────────────────────────────────────────────────────────
lat_rows = []
for name, df in dfs.items():
    lat = df['latency_ms']
    lat_rows.append({'Model': name,
                     'Mean (ms)': round(lat.mean(), 0),
                     'Median (ms)': round(lat.median(), 0),
                     'p95 (ms)': round(lat.quantile(0.95), 0),
                     'Throughput (t/s)': round(1000/lat.mean(), 2)})
lat_df = pd.DataFrame(lat_rows).set_index('Model').loc[MODEL_ORDER]
display(lat_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(MODEL_ORDER)); w = 0.35
ax = axes[0]
br = ax.bar(x-w/2, lat_df['Mean (ms)'],  w, color=[MODEL_COLORS[m] for m in MODEL_ORDER], alpha=0.9, label='Mean', edgecolor='white')
bp = ax.bar(x+w/2, lat_df['p95 (ms)'],   w, color=[MODEL_COLORS[m] for m in MODEL_ORDER], alpha=0.5, label='p95',  edgecolor='white', hatch='//')
for bar in list(br)+list(bp):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, rotation=12, fontsize=9)
ax.set_ylabel('Latency (ms)'); ax.set_title('Inference Latency — Mean & p95', fontweight='bold')
ax.legend()

ax = axes[1]
bp2 = ax.boxplot([dfs[m]['latency_ms'].values for m in MODEL_ORDER], patch_artist=True,
                 medianprops=dict(color='navy', lw=2))
for patch, name in zip(bp2['boxes'], MODEL_ORDER):
    patch.set_facecolor(MODEL_COLORS[name]); patch.set_alpha(0.7)
ax.set_xticklabels(MODEL_ORDER, rotation=12, fontsize=9)
ax.set_ylabel('Latency (ms)'); ax.set_title('Latency Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT / 'latency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── GPU VRAM ──────────────────────────────────────────────────────────────────
def clean_vram(path, model_name):
    df = pd.read_csv(path)
    mean_col = [c for c in df.columns if 'memoryAllocated' in c and 'MIN' not in c and 'MAX' not in c][0]
    max_col  = [c for c in df.columns if 'MAX' in c][0]
    out = df[['Relative Time (Process)', mean_col, max_col]].copy()
    out.columns = ['time_s','vram_bytes','vram_bytes_max']
    out['vram_gb']     = out['vram_bytes']     / 1e9
    out['vram_gb_max'] = out['vram_bytes_max'] / 1e9
    out['model'] = model_name
    return out

vl = clean_vram(RES / 'GPU_VRAM_inference_llama.csv',   'LLaMA Fine-tuned')
vm = clean_vram(RES / 'GPU_VRAM_inference_mistral.csv', 'Mistral Fine-tuned')
vram = pd.concat([vl, vm], ignore_index=True)

vram_summary = vram.groupby('model').agg(
    peak_vram_gb=('vram_gb_max','max'),
    mean_vram_gb=('vram_gb','mean')
).round(2)
display(vram_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for sub in [vl, vm]:
    name = sub['model'].iloc[0]
    ax.plot(sub['time_s'], sub['vram_gb'], label=name, color=MODEL_COLORS[name], lw=2)
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('VRAM allocated (GB)')
ax.set_title('VRAM Over Time', fontweight='bold'); ax.legend()

ax = axes[1]
m_names = vram_summary.index.tolist()
bars = ax.bar(m_names, vram_summary['peak_vram_gb'],
              color=[MODEL_COLORS[m] for m in m_names], edgecolor='white', width=0.5)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{bar.get_height():.1f} GB', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Peak VRAM (GB)'); ax.set_title('Peak GPU VRAM at Inference', fontweight='bold')
ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig(OUT / 'gpu_vram.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Comparison III: Student vs. Human (Gold Standard, n=50)

In [ ]:
# ── Accuracy by level vs human gold ──────────────────────────────────────────
def gold_row(df, vs_prefix, name):
    return {
        'Model': name,
        'L1':    round(df[f'{vs_prefix}_l1'].mean(),   4),
        'L2':    round(df[f'{vs_prefix}_l2'].mean(),   4),
        'L3':    round(df[f'{vs_prefix}_l3'].mean(),   4),
        'Full':  round(df[f'{vs_prefix}_full'].mean(), 4),
    }

gold_acc = pd.DataFrame([
    gold_row(poc,  'vs_human', 'Teacher (PoC)'),
    gold_row(gold['LLaMA Fine-tuned'],   'vs_human', 'LLaMA Fine-tuned'),
    gold_row(gold['Mistral Fine-tuned'], 'vs_human', 'Mistral Fine-tuned'),
]).set_index('Model')

print('=== Accuracy vs Human Gold (n=50) ===')
display(gold_acc)

In [ ]:
# ── Cohen's κ — student vs human, teacher vs human ────────────────────────────
print('=== Cohen\'s κ (Full label) vs Human ===')
kappa_rows = []
for name, df, pred_col in [
    ('Teacher (PoC)',       poc,                          'label'),
    ('LLaMA Fine-tuned',   gold['LLaMA Fine-tuned'],    'predicted_label'),
    ('Mistral Fine-tuned', gold['Mistral Fine-tuned'],  'predicted_label'),
]:
    merged = pd.DataFrame({'pred': df[pred_col], 'human': df['human_label']}).dropna()
    k   = cohen_kappa_score(merged['pred'].astype(str), merged['human'].astype(str))
    acc = (merged['pred'].astype(str) == merged['human'].astype(str)).mean()
    kappa_rows.append({'Model': name,
                       'Accuracy vs Human': round(acc, 4),
                       'Cohen κ': round(k, 4),
                       'n': len(merged)})
    print(f'{name:25s}  acc={acc:.3f}  κ={k:.3f}  (n={len(merged)})')

kappa_df = pd.DataFrame(kappa_rows).set_index('Model')
print('\nHuman inter-rater κ (reference from §5.4): 0.434')
display(kappa_df)

In [ ]:
# ── Visual III-1: Gold accuracy + Cohen's κ ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gold_models = ['Teacher (PoC)', 'LLaMA Fine-tuned', 'Mistral Fine-tuned']

ax = axes[0]
gold_acc.plot.bar(ax=ax, color=_PAL[:4], edgecolor='white', width=0.7)
ax.set_title('Accuracy vs Human Gold — by Level\n(n=50 gold tickets)', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.tick_params(axis='x', rotation=12)
ax.legend(title='Level')
for bar in ax.patches:
    h = bar.get_height()
    if h > 0.02:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.0%}',
                ha='center', va='bottom', fontsize=8)

ax = axes[1]
kappa_df[['Accuracy vs Human','Cohen κ']].plot.bar(
    ax=ax, color=_PAL[:2], edgecolor='white', width=0.6)
ax.axhline(0.434, color='grey', lw=1.5, ls='--', label='Human inter-rater κ = 0.434')
ax.set_title('Accuracy & Cohen\'s κ vs Human\n(full label, n=50)', fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=12)
ax.legend(fontsize=9)
for bar in ax.patches:
    h = bar.get_height()
    if h > 0.02:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUT / 'gold_standard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error typology on gold set (mirroring §7.2 analysis) ─────────────────────
for name in ['LLaMA Fine-tuned', 'Mistral Fine-tuned']:
    gdf = gold[name].dropna(subset=['human_label'])
    wrong = gdf[~gdf['vs_human_full']]
    n = len(wrong)
    if n == 0:
        print(f'{name}: no errors vs human gold')
        continue
    gran = (wrong['vs_human_l1'] & wrong['vs_human_l2'] & ~wrong['vs_human_l3']).sum()
    sub  = (wrong['vs_human_l1'] & ~wrong['vs_human_l2']).sum()
    cross= (~wrong['vs_human_l1']).sum()
    fb   = (wrong['pred_l1'] == '1 Misc incidents').sum()
    print(f'\n=== Error typology vs Human — {name} (n_wrong={n}) ===')
    print(f'  Granular symptom (L1✓ L2✓ L3✗): {gran}  ({gran/n:.0%})')
    print(f'  Subcategory      (L1✓ L2✗):     {sub}   ({sub/n:.0%})')
    print(f'  Cross-domain     (L1✗):          {cross}  ({cross/n:.0%})')
    print(f'  Fallback retreat (→Misc):        {fb}   ({fb/n:.0%})')

---
## Summary Tables (Export)

In [ ]:
# ── Master summary table ──────────────────────────────────────────────────────
print('=== MASTER RESULTS TABLE ===')
display(metrics_table)

# Export all tables
metrics_table.to_csv(OUT / 'table_main_results.csv')
gold_acc.to_csv(OUT / 'table_gold_accuracy.csv')
kappa_df.to_csv(OUT / 'table_gold_kappa.csv')
f1_all.to_csv(OUT / 'table_per_category_f1.csv', index=False)
gen_quality.to_csv(OUT / 'table_generation_quality.csv')
lat_df.to_csv(OUT / 'table_latency.csv')
vram_summary.to_csv(OUT / 'table_vram.csv')
print('All CSV tables exported.')